# wwd_i — Phase 2: train the backbone on Colab (T4)

Trains the from-scratch **BC-ResNet** backbone with **cosine-prototypical** metric
learning on Speech Commands v2, runs the **few-shot probe gate** on held-out
(unseen) words, and exports a frozen `backbone.onnx`.

**Before running:** *Runtime → Change runtime type → **T4 GPU***.

This run stays at project parity — **Python 3.14 + torch 2.12** via `uv` — but
installs the **CUDA** build of torch for the T4 (local dev pins the CPU build).


### 1. Confirm the T4 GPU


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

### 2. Get the code

This `git clone`s the repo, so push `wwd_i` to GitHub first (private is fine — for
a private repo use `https://<TOKEN>@github.com/<you>/wwd_i.git`). No remote? Skip
this cell and use **2b. Upload a zip** below instead.


In [ ]:
import os

REPO_URL = "https://github.com/<you>/wwd_i.git"  # <-- set me
BRANCH = "master"
os.environ["REPO_URL"] = REPO_URL
os.environ["BRANCH"] = BRANCH
!rm -rf /content/wwd_i && git clone --branch "$BRANCH" --depth 1 "$REPO_URL" /content/wwd_i
%cd /content/wwd_i
!git log --oneline -1

### 2b. Alternative — upload a zip (no Git remote)

Locally run `git archive -o /tmp/wwd_i.zip HEAD`, then run this instead of cell 2:

```python
from google.colab import files
files.upload()                          # choose wwd_i.zip
!rm -rf /content/wwd_i && mkdir -p /content/wwd_i && unzip -q wwd_i.zip -d /content/wwd_i
%cd /content/wwd_i
```


### 3. Provision Python 3.14 with `uv`


In [ ]:
!pip install -q uv
!uv venv --python 3.14
!.venv/bin/python --version

### 4. Install CUDA torch (override the CPU pin) + the package

`--no-config` ignores the repo's CPU torch pin; `--torch-backend=auto` detects the
T4's CUDA and selects matching wheels. If `auto` ever fails, swap it for
`--index-url https://download.pytorch.org/whl/cu124`.


In [ ]:
!uv pip install --no-config --python .venv/bin/python torch torchaudio --torch-backend=auto
# our package (pulls numpy/onnx/onnxruntime/soundfile/soxr) + the dynamo ONNX exporter dep
!uv pip install --python .venv/bin/python -e . onnxscript

### 5. Verify CUDA torch is active


In [ ]:
!.venv/bin/python -c "import torch; assert torch.cuda.is_available(), 'CUDA torch NOT active - is the runtime set to T4?'; print('OK:', torch.__version__, '|', torch.cuda.get_device_name(0))"

### 6. Quick smoke run

End-to-end check on a small slice. The **first** run downloads Speech Commands v2
(~2.3 GB) into `/content/data` — a few minutes. Confirms GPU training works and the
probe prints before committing to the full run.


In [ ]:
!.venv/bin/python -m wwd_i.train.train_backbone --device cuda --root /content/data/speech_commands --limit 80 --steps 150 --eval-every 75 --probe-episodes 30 --n-way 10 --probe-n-way 10 --k-shot 5 --q-query 5 --out /content/artifacts/backbone_smoke.pt --onnx /content/artifacts/backbone_smoke.onnx

### 7. Full run — the Phase-2 gate

Watch the `[probe @ N]` lines: backbone accuracy should climb **clearly above** the
mel-baseline (a positive, growing `margin`). That margin on held-out words is the
evidence the frozen embedding generalizes to words it never trained on. Tune
`--steps`, `--n-way`, `--embedding-dim` as needed; this is the expected iteration phase.


In [ ]:
!.venv/bin/python -m wwd_i.train.train_backbone --device cuda --root /content/data/speech_commands --steps 4000 --eval-every 500 --probe-episodes 200 --n-way 12 --probe-n-way 10 --k-shot 5 --q-query 5 --lr 1e-3 --embedding-dim 96 --out /content/artifacts/backbone.pt --onnx /content/artifacts/backbone.onnx

### 8. Download the artifacts


In [ ]:
from google.colab import files

files.download("/content/artifacts/backbone.onnx")
files.download("/content/artifacts/backbone.pt")

### Interpreting the gate

- **Pass:** backbone probe accuracy is clearly above the mel-baseline on held-out
  words → the embedding generalizes → Phase 2 is satisfied. Drop `backbone.onnx`
  into the repo and move on to Phase 3 (per-word data) / Phase 4 (head).
- **Weak margin:** widen the backbone (`--embedding-dim`, or edit `BackboneConfig`
  channels) and/or train longer. Speech Commands is a small-vocab bootstrap; the
  full transfer story scales with a larger corpus (MSWC) using the same pipeline.
